# Forecasting de ventas 2025
## 1. Librerías y carga de datos de inferencia

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import holidays
import joblib

pd.set_option("display.max_columns", None)

# Cargar los datos de 2025 para inferencia
inferencia_path = "../data/raw/inferencia/ventas_2025_inferencia.csv"
inferencia_df = pd.read_csv(inferencia_path, parse_dates=["fecha"])

print(f"inferencia_df.shape: {inferencia_df.shape}")
inferencia_df.head()

inferencia_df.shape: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [2]:
# Rango de fechas y ventas por mes en el archivo de inferencia
print(f"Desde {inferencia_df['fecha'].min().date()} hasta {inferencia_df['fecha'].max().date()}")
print(f"Valores vacíos en unidades_vendidas: {inferencia_df['unidades_vendidas'].isna().sum()}")

inferencia_df.groupby(inferencia_df["fecha"].dt.month)["unidades_vendidas"].agg(["count", "mean"]).round(2)

Desde 2025-10-25 hasta 2025-11-30
Valores vacíos en unidades_vendidas: 720


,count,mean
fecha,,
10,168,4.67
11,0,NaN


## 2. Variables temporales y de calendario
Añade las variables de calendario usadas por el modelo, con Black Friday calculado a partir del cuarto jueves de noviembre.

In [3]:
# Asegura que fecha tenga formato datetime antes de derivar el calendario.
inferencia_df["fecha"] = pd.to_datetime(inferencia_df["fecha"])

# Calcula festivos nacionales de España para los años presentes.
anios_holidays = sorted(inferencia_df["fecha"].dt.year.dropna().unique().tolist())
festivos_es = holidays.country_holidays("ES", years=[int(anio) for anio in anios_holidays])
festivos_normales = pd.to_datetime(list(festivos_es.keys())).normalize()

# Black Friday es el día después del cuarto jueves; Cyber Monday llega tres días después.
fechas_black_friday = {}
for anio in anios_holidays:
    dias_noviembre = pd.date_range(
        start=pd.Timestamp(year=int(anio), month=11, day=1),
        end=pd.Timestamp(year=int(anio), month=11, day=30),
        freq="D",
    )
    jueves_noviembre = dias_noviembre[dias_noviembre.dayofweek == 3]
    fechas_black_friday[int(anio)] = jueves_noviembre[3] + pd.Timedelta(days=1)

black_friday_por_anio = pd.Series(
    {anio: pd.Timestamp(fecha).normalize() for anio, fecha in fechas_black_friday.items()}
)
cyber_monday_por_anio = black_friday_por_anio + pd.Timedelta(days=3)

# Deriva las mismas variables temporales que se crearon durante el entrenamiento.
inferencia_df["año"] = inferencia_df["fecha"].dt.year
inferencia_df["mes"] = inferencia_df["fecha"].dt.month
inferencia_df["dia_mes"] = inferencia_df["fecha"].dt.day
inferencia_df["dia_semana"] = inferencia_df["fecha"].dt.dayofweek
inferencia_df["trimestre"] = inferencia_df["fecha"].dt.quarter
inferencia_df["semana_año"] = inferencia_df["fecha"].dt.isocalendar().week.astype("int16")
inferencia_df["es_fin_semana"] = inferencia_df["fecha"].dt.dayofweek.isin([5, 6]).astype("int8")
inferencia_df["es_festivo"] = inferencia_df["fecha"].dt.normalize().isin(festivos_normales).astype("int8")
inferencia_df["es_laborable"] = (
    ~inferencia_df["es_fin_semana"].astype(bool)
    & ~inferencia_df["es_festivo"].astype(bool)
).astype("int8")
inferencia_df["es_inicio_mes"] = inferencia_df["fecha"].dt.day.le(3).astype("int8")
inferencia_df["es_fin_mes"] = inferencia_df["fecha"].dt.day.ge(
    inferencia_df["fecha"].dt.days_in_month - 2
).astype("int8")
inferencia_df["es_Black_Friday"] = (
    inferencia_df["año"].map(black_friday_por_anio)
    .eq(inferencia_df["fecha"].dt.normalize())
).astype("int8")
inferencia_df["es_Cyber_Monday"] = (
    inferencia_df["año"].map(cyber_monday_por_anio)
    .eq(inferencia_df["fecha"].dt.normalize())
).astype("int8")


## 3. Lags y media móvil de unidades vendidas
Calcula lags de 1 a 7 días y la media de los siete días previos por producto y año, sin eliminar nulos.

In [4]:
# Ordena las observaciones para que los desplazamientos sigan el historial cronológico.
inferencia_df.sort_values(["producto_id", "fecha"], inplace=True)

# Mantiene separados los historiales de cada producto y año.
grupos_producto_anio = inferencia_df.groupby(
    ["producto_id", inferencia_df["fecha"].dt.year]
)["unidades_vendidas"]

# Crea los lags de 1 a 7, conservando los valores ausentes para el cálculo recursivo.
for lag in range(1, 8):
    inferencia_df[f"unidades_vendidas_lag{lag}"] = grupos_producto_anio.shift(lag)

# Calcula la media móvil sobre los siete valores previos, sin incluir el día actual.
inferencia_df["unidades_vendidas_ma7"] = grupos_producto_anio.transform(
    lambda valores: valores.shift(1).rolling(window=7, min_periods=7).mean()
)


## 4. Descuento y precios de competencia
Deriva las variables de precio del entrenamiento y elimina las columnas de competidores originales.

In [5]:
# Calcula el descuento porcentual con la misma fórmula usada en entrenamiento.
inferencia_df["descuento_porcentaje"] = (
    (inferencia_df["precio_venta"] - inferencia_df["precio_base"])
    / inferencia_df["precio_base"]
    * 100
)

# Promedia los precios de los competidores y calcula el ratio frente al precio de venta.
inferencia_df["precio_competencia"] = inferencia_df[
    ["Amazon", "Decathlon", "Deporvillage"]
].mean(axis=1)
inferencia_df["ratio_precio"] = (
    inferencia_df["precio_venta"] / inferencia_df["precio_competencia"]
)

# Retira las columnas originales de competencia tras crear las variables derivadas.
inferencia_df.drop(
    columns=["Amazon", "Decathlon", "Deporvillage"],
    inplace=True,
)


## 5. One Hot Encoding y columnas del modelo
Crea las copias categóricas con sufijo `_h`, codifica las categorías y completa las variables requeridas por el modelo.

In [6]:
# Duplica las variables categóricas con los nombres usados en entrenamiento.
inferencia_df["nombre_h"] = inferencia_df["nombre"]
inferencia_df["categoria_h"] = inferencia_df["categoria"]
inferencia_df["subcategoria_h"] = inferencia_df["subcategoria"]

# Aplica One Hot Encoding sin descartar ninguna categoría observada.
inferencia_df = pd.get_dummies(
    inferencia_df,
    columns=["nombre_h", "categoria_h", "subcategoria_h"],
    drop_first=False,
)

# Carga el modelo y añade en False las columnas predictoras que no aparezcan en inferencia.
modelo_final = joblib.load("../models/modelo_final.joblib")
columnas_modelo = modelo_final.feature_names_in_.tolist()
columnas_faltantes_modelo = [
    columna for columna in columnas_modelo if columna not in inferencia_df.columns
]
for columna in columnas_faltantes_modelo:
    inferencia_df[columna] = False


## 6. Filtrado a noviembre y exportación
Conserva noviembre, guarda el DataFrame transformado y muestra la forma final y las columnas del modelo ausentes.

In [7]:
# Conserva únicamente noviembre después de generar las variables del calendario.
inferencia_df = inferencia_df[inferencia_df["mes"] == 11].copy()

# Verifica que no falte ninguna columna esperada por el modelo.
columnas_modelo_ausentes = [
    columna for columna in columnas_modelo if columna not in inferencia_df.columns
]

# Guarda el conjunto de inferencia procesado en el directorio de datos procesados.
inferencia_df.to_csv(
    "../data/processed/inferencia_df_transformado.csv",
    index=False,
)

# Muestra las comprobaciones solicitadas.
print(f"Shape final de inferencia_df: {inferencia_df.shape}")
print(f"Columnas del modelo faltantes: {len(columnas_modelo_ausentes)}")
inferencia_df.head()


Shape final de inferencia_df: (720, 78)
Columnas del modelo faltantes: 0


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,año,mes,dia_mes,dia_semana,trimestre,semana_año,es_fin_semana,es_festivo,es_laborable,es_inicio_mes,es_fin_mes,es_Black_Friday,es_Cyber_Monday,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_ma7,descuento_porcentaje,precio_competencia,ratio_precio,nombre_h_Adidas Own The Run Jacket,nombre_h_Adidas Ultraboost 23,nombre_h_Asics Gel Nimbus 25,nombre_h_Bowflex SelectTech 552,nombre_h_Columbia Silver Ridge,nombre_h_Decathlon Bandas Elásticas Set,nombre_h_Domyos BM900,nombre_h_Domyos Kit Mancuernas 20kg,nombre_h_Gaiam Premium Yoga Block,nombre_h_Liforme Yoga Pad,nombre_h_Lotuscrafts Yoga Bolster,nombre_h_Manduka PRO Yoga Mat,nombre_h_Merrell Moab 2 GTX,nombre_h_New Balance Fresh Foam X 1080v12,nombre_h_Nike Air Zoom Pegasus 40,nombre_h_Nike Dri-FIT Miler,nombre_h_Puma Velocity Nitro 2,nombre_h_Quechua MH500,nombre_h_Reebok Floatride Energy 5,nombre_h_Reebok Professional Deck,nombre_h_Salomon Speedcross 5 GTX,nombre_h_Sveltus Kettlebell 12kg,nombre_h_The North Face Borealis,nombre_h_Trek Marlin 7,categoria_h_Fitness,categoria_h_Outdoor,categoria_h_Running,categoria_h_Wellness,subcategoria_h_Banco Gimnasio,subcategoria_h_Bandas Elásticas,subcategoria_h_Bicicleta Montaña,subcategoria_h_Bloque Yoga,subcategoria_h_Cojín Yoga,subcategoria_h_Esterilla Fitness,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,11,1,5,4,44,1,1,0,1,0,0,0,14.0,10.0,14.0,15.0,16.0,20.0,26.0,16.428571,0.0,100.610000,1.143028,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
192,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,11,2,6,4,44,1,0,0,1,0,0,0,NaN,14.0,10.0,14.0,15.0,16.0,20.0,NaN,0.0,97.740000,1.176591,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
216,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,11,3,0,4,45,0,0,1,1,0,0,0,NaN,NaN,14.0,10.0,14.0,15.0,16.0,NaN,0.0,99.513333,1.155624,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
240,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,11,4,1,4,45,0,0,1,0,0,0,0,NaN,NaN,NaN,14.0,10.0,14.0,15.0,NaN,0.0,105.553333,1.089497,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
264,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,11,5,2,4,45,0,0,1,0,0,0,0,NaN,NaN,NaN,NaN,14.0,10.0,14.0,NaN,0.0,100.276667,1.146827,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,

# Controles de la transformación de inferencia

In [8]:
# Controles de la transformación de inferencia
print(f"Shape: {inferencia_df.shape}")
print(f"Fechas: {inferencia_df['fecha'].min().date()} a {inferencia_df['fecha'].max().date()}")
print("Black Friday marcado:", inferencia_df.loc[inferencia_df["es_Black_Friday"] == 1, "fecha"].dt.date.unique())

modelo_final = joblib.load("../models/modelo_final.joblib")
faltantes = [c for c in modelo_final.feature_names_in_ if c not in inferencia_df.columns]
print(f"Columnas del modelo que faltan: {len(faltantes)}")

# Lags del 1 de noviembre (deben estar completos)
inferencia_df.loc[
    inferencia_df["fecha"] == "2025-11-01",
    ["producto_id", "unidades_vendidas_lag1", "unidades_vendidas_lag7", "unidades_vendidas_ma7"],
].head()

Shape: (720, 78)
Fechas: 2025-11-01 a 2025-11-30
Black Friday marcado: [datetime.date(2025, 11, 28)]
Columnas del modelo que faltan: 0


,producto_id,unidades_vendidas_lag1,unidades_vendidas_lag7,unidades_vendidas_ma7
168,PROD_001,14.0,26.0,16.428571
169,PROD_002,13.0,27.0,16.142857
170,PROD_003,2.0,5.0,4.714286
171,PROD_004,3.0,3.0,4.000000
172,PROD_005,4.0,3.0,3.285714


In [11]:
# Cuenta los identificadores de producto únicos.
cantidad_productos = inferencia_df["producto_id"].nunique()

print(f"Cantidad de productos distintos: {cantidad_productos}")

Cantidad de productos distintos: 24


In [12]:
# Obtiene y ordena las fechas únicas del dataset.
fechas_inferencia = inferencia_df["fecha"].drop_duplicates().sort_values()

print(f"Cantidad de fechas: {len(fechas_inferencia)}")
print(fechas_inferencia.to_string(index=False))

Cantidad de fechas: 30
2025-11-01
2025-11-02
2025-11-03
2025-11-04
2025-11-05
2025-11-06
2025-11-07
2025-11-08
2025-11-09
2025-11-10
2025-11-11
2025-11-12
2025-11-13
2025-11-14
2025-11-15
2025-11-16
2025-11-17
2025-11-18
2025-11-19
2025-11-20
2025-11-21
2025-11-22
2025-11-23
2025-11-24
2025-11-25
2025-11-26
2025-11-27
2025-11-28
2025-11-29
2025-11-30


In [13]:
# Resumen final de inferencia_df transformado
print(f"Total de registros: {len(inferencia_df)}")
print(f"Total de columnas: {len(inferencia_df.columns)}")
print(f"Productos únicos: {inferencia_df['producto_id'].nunique()}")
print(f"Rango de fechas: {inferencia_df['fecha'].min().date()} a {inferencia_df['fecha'].max().date()}")

Total de registros: 720
Total de columnas: 78
Productos únicos: 24
Rango de fechas: 2025-11-01 a 2025-11-30
